<div style="padding: 28px 32px; border: 1px solid #bfdbfe; border-left: 8px solid #2563eb; border-radius: 12px; background: linear-gradient(135deg, #eff6ff 0%, #f8fafc 100%);">

<h1 style="margin: 0; color: #1e3a8a;">Machine Learning Zoomcamp 2026</h1>
<p style="margin: 8px 0 18px; font-size: 1.2em; color: #334155;"><strong>Homework 6</strong> · Decision Trees and Ensemble Learning</p>

| Student | Cohort | Dataset | Environment |
| :--- | :--- | :--- | :--- |
| Carlos Ibarra | 2026 | 2026 Car Fuel Efficiency | Python 3.11+ · Pandas · Scikit-Learn · XGBoost |

</div>

### Learning path

| Step | Focus |
| :---: | :--- |
| 01 | Initial setup, dataset integrity check, and data loading |
| 02 | Missing-value preparation, data splitting, and vectorization |
| 03 | Q1: Decision tree split |
| 04 | Q2: Random forest baseline |
| 05 | Q3: Number of trees |
| 06 | Q4: Tree depth |
| 07 | Q5: Feature importance |
| 08 | Q6: XGBoost learning rate |
| 09 | Results dashboard and summary |

<div style="padding: 18px 22px; border-left: 6px solid #2563eb; border-radius: 8px; background: #eff6ff;">

## Step 1 · Initial setup and data loading

**Objective.** Import required libraries, define the fixed parameters of the assignment, verify the SHA-256 checksum of the 2026 car fuel efficiency CSV, and load it into a Pandas DataFrame.

</div>

In [1]:
# Import hashlib and pathlib for the integrity check and path resolution.
import hashlib
import platform
import sys
from pathlib import Path

import sklearn

# Import numerical and tabular analysis libraries.
import numpy as np
import pandas as pd

# Import Scikit-Learn tools for splitting and encoding.
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer

# Record the actual notebook interpreter and numerical library versions.
print(f"Python: {platform.python_version()}")
print(f"Interpreter: {sys.executable}")
print(f"NumPy: {np.__version__}; pandas: {pd.__version__}; scikit-learn: {sklearn.__version__}")

# Define fixed constants according to the HW6 specification.
DATA_PATH = Path("data/car_fuel_efficiency_2026.csv")
EXPECTED_SHA256 = "00a5cab178a8b7cd6e9157ee71dabc236ba7f20b1832336d358b07af14ba431f"
TARGET = "fuel_efficiency_mpg"
RANDOM_STATE = 1

# Confirm the file is the exact 2026 cohort release before using it.
actual_sha256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
assert actual_sha256 == EXPECTED_SHA256, f"Unexpected checksum: {actual_sha256}"
print(f"SHA-256 verified: {actual_sha256}")

# Load dataset.
df = pd.read_csv(DATA_PATH)

# Display dimensions and preview.
print(f"Dataset shape: {df.shape}")
df.head()

Python: 3.14.6
Interpreter: /Users/carlosibarra/projects/machine-learning-zoomcamp-homework/.venv/bin/python
NumPy: 2.4.6; pandas: 3.0.6; scikit-learn: 1.9.1
SHA-256 verified: 00a5cab178a8b7cd6e9157ee71dabc236ba7f20b1832336d358b07af14ba431f
Dataset shape: (10000, 11)


,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [2]:
# Inspect column types, missing values, and cardinality before any preparation.
inspection = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "unique": df.nunique(),
})
display(inspection)

# Summarize the target variable.
print(f"Target '{TARGET}' summary:")
display(df[TARGET].describe().to_frame().T)

,dtype,missing,unique
model_year,int64,0,50
origin,str,0,3
fuel_type,str,0,3
drivetrain,str,0,3
num_doors,int64,0,4
engine_displacement,int64,0,127
num_cylinders,int64,0,5
horsepower,float64,877,153
vehicle_weight,int64,0,176
acceleration,float64,264,93


Target 'fuel_efficiency_mpg' summary:


,count,mean,std,min,25%,50%,75%,max
fuel_efficiency_mpg,10000.0,29.9977,2.930245,19.8,28.0,30.0,31.9,41.2


<div style="padding: 14px 18px; border-left: 5px solid #16a34a; border-radius: 8px; background: #f0fdf4;">


**Result · Dataset overview**

The file matches the documented SHA-256 checksum and contains **`10,000`** observations and **`11`** columns. The target `fuel_efficiency_mpg` has no missing values (mean `29.998`, standard deviation `2.930`, range `19.8` to `41.2`). Missing values appear only in two predictors: **`horsepower`** (`877`) and **`acceleration`** (`264`). Three predictors are categorical (`origin`, `fuel_type`, `drivetrain`); the remaining seven are numerical.

</div>

<div style="padding: 18px 22px; border-left: 6px solid #2563eb; border-radius: 8px; background: #eff6ff;">

## Step 2 · Missing-value preparation, data splitting, and vectorization

**Objective.** Fill missing predictor values with zeros as prescribed by this homework, split the data into 60% train, 20% validation, and 20% test with the exact two-step `train_test_split` calls (`random_state=1`), separate the target without changing row alignment, and encode the predictors with `DictVectorizer(sparse=True)`, fitting it on the training records only.

</div>

In [3]:
# Select predictor columns so the target is never modified.
feature_columns = df.columns.drop(TARGET)
target_before = df[TARGET].copy()

# Record missing values before filling them.
missing_before = df[feature_columns].isna().sum()

# Fill missing predictor values with zeros, as prescribed by this homework.
df[feature_columns] = df[feature_columns].fillna(0)

pd.testing.assert_series_equal(df[TARGET], target_before)
print("Missing values before filling:")
print(missing_before[missing_before > 0])
print(f"\nMissing values remaining: {int(df.isna().sum().sum())}")
print(f"Target unchanged: {df[TARGET].equals(target_before)}")

Missing values before filling:
horsepower      877
acceleration    264
dtype: int64

Missing values remaining: 0
Target unchanged: True


In [4]:
# Split with the assignment's exact calls: 80/20, then 75/25 of full train.
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)

# Give each partition its own consecutive index.
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

# Save the targets before removing them from the predictor frames.
y_train = df_train[TARGET].values
y_val = df_val[TARGET].values
y_test = df_test[TARGET].values

df_train = df_train.drop(columns=TARGET)
df_val = df_val.drop(columns=TARGET)
df_test = df_test.drop(columns=TARGET)

print(f"Full train before the second split: {df_full_train.shape[0]} rows")
print(f"Train: {df_train.shape}, validation: {df_val.shape}, test: {df_test.shape}")
print(f"Target lengths: {len(y_train)}, {len(y_val)}, {len(y_test)}")

Full train before the second split: 8000 rows
Train: (6000, 10), validation: (2000, 10), test: (2000, 10)
Target lengths: 6000, 2000, 2000


In [5]:
# Convert each predictor frame into a list of dictionaries, one per row.
train_dicts = df_train.to_dict(orient="records")
val_dicts = df_val.to_dict(orient="records")
test_dicts = df_test.to_dict(orient="records")

# Learn the feature mapping from the training records only.
dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(train_dicts)

# Reuse the fitted mapping for validation and test records.
X_val = dv.transform(val_dicts)
X_test = dv.transform(test_dicts)

feature_names = dv.get_feature_names_out()
print(f"X_train: {X_train.shape}, X_val: {X_val.shape}, X_test: {X_test.shape}")
print(f"Matrix format: {X_train.format.upper()} (sparse)")
print(f"\nFeatures ({len(feature_names)}):")
print(list(feature_names))

X_train: (6000, 16), X_val: (2000, 16), X_test: (2000, 16)
Matrix format: CSR (sparse)

Features (16):
['acceleration', 'drivetrain=All-wheel drive', 'drivetrain=Front-wheel drive', 'drivetrain=Rear-wheel drive', 'engine_displacement', 'fuel_type=Diesel', 'fuel_type=Gasoline', 'fuel_type=Hybrid', 'horsepower', 'model_year', 'num_cylinders', 'num_doors', 'origin=Asia', 'origin=Europe', 'origin=USA', 'vehicle_weight']


In [6]:
# Define RMSE: the square root of the mean squared error, in MPG units.
def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))


# Sanity check: errors of 3 and 4 give sqrt((9 + 16) / 2).
assert np.isclose(rmse(np.array([0.0, 0.0]), np.array([3.0, 4.0])), np.sqrt(12.5))

<div style="padding: 14px 18px; border-left: 5px solid #16a34a; border-radius: 8px; background: #f0fdf4;">


**Result · Missing values, split, and vectorization**

| Column | Missing values before filling | Replacement |
| :--- | ---: | :--- |
| `horsepower` | 877 | `0` |
| `acceleration` | 264 | `0` |

No missing values remain, and `fuel_efficiency_mpg` is unchanged. The two splits yield **6,000 training**, **2,000 validation**, and **2,000 test** rows (`df_full_train` held 8,000 rows before the second split). Each predictor frame has **10** columns after removing the target.

`DictVectorizer(sparse=True)` learns **16** features from the training records only: 7 numerical columns plus 9 one-hot columns (3 each for `origin`, `fuel_type`, and `drivetrain`). Validation and test records are only transformed. All matrices are sparse CSR with 16 columns, and the test set stays reserved for the final evaluation.

Zero is a placeholder required by the assignment, not a measured value: a `horsepower` or `acceleration` of `0` really means "unknown" for those rows.

</div>

<div style="padding: 18px 22px; border-left: 6px solid #2563eb; border-radius: 8px; background: #eff6ff;">

## Step 3 · Question 1: Decision tree split

**Objective.** Train a `DecisionTreeRegressor` with `max_depth=1` on the training matrix, inspect its single split, and map the one-hot feature name back to its original dataset column. Options: `vehicle_weight`, `model_year`, `origin`, `fuel_type`.

</div>

In [7]:
# Import the decision tree regressor and a text exporter for its rules.
from sklearn.tree import DecisionTreeRegressor, export_text

# Train a single-split tree (decision stump) on the training matrix.
dt = DecisionTreeRegressor(max_depth=1)
dt.fit(X_train, y_train)

# Show the learned rule using the vectorizer's feature names.
print(export_text(dt, feature_names=list(feature_names)))

# Read the root node: column index, threshold, and the matching feature name.
root_index = dt.tree_.feature[0]
root_threshold = dt.tree_.threshold[0]
root_feature = feature_names[root_index]
print(f"Root split: {root_feature} <= {root_threshold:.4f}")

# Summarize the two leaves: number of training rows and predicted mean MPG.
for side, node in [
    ("left (condition true)", dt.tree_.children_left[0]),
    ("right (condition false)", dt.tree_.children_right[0]),
]:
    n_rows = dt.tree_.n_node_samples[node]
    mean_mpg = dt.tree_.value[node][0][0]
    print(f"{side}: {n_rows} rows, mean MPG {mean_mpg:.3f}")

# Map a one-hot name such as "origin=USA" back to its original column.
original_column = root_feature.split("=")[0]
assert original_column in df_train.columns
print(f"Original column: {original_column}")

|--- model_year <= 1997.50
|   |--- value: [28.61]
|--- model_year >  1997.50
|   |--- value: [31.17]

Root split: model_year <= 1997.5000
left (condition true): 2716 rows, mean MPG 28.614
right (condition false): 3284 rows, mean MPG 31.168
Original column: model_year


<div style="padding: 14px 18px; border-left: 5px solid #16a34a; border-radius: 8px; background: #f0fdf4;">

**Result · Question 1 (decision tree split)**

A depth-1 tree (decision stump) picks the single column and threshold that minimize the squared error inside the two resulting groups. Its split is **`model_year <= 1997.5`**. It separates **`2,716`** training rows (model year 1997 or earlier, mean `28.614` MPG) from **`3,284`** rows (1998 or later, mean `31.168` MPG), so newer cars average about `2.55` MPG more.

The selected feature is a numerical column, so no one-hot name (such as `origin=USA`) needed to be mapped back to an original column.

**Selected option:** **`model_year`**
</div>

<div style="padding: 18px 22px; border-left: 6px solid #2563eb; border-radius: 8px; background: #eff6ff;">

## Step 4 · Question 2: Random forest baseline

**Objective.** Train a `RandomForestRegressor` with `n_estimators=10` and `random_state=1` (`n_jobs=-1` is optional), and compute its validation RMSE. Options: `0.1837`, `1.837`, `18.37`, `183.7`.

</div>

In [8]:
# Import the random forest regressor.
from sklearn.ensemble import RandomForestRegressor

# Train a 10-tree forest on the training matrix.
rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)

# Predict the validation targets and compute RMSE in MPG units.
y_pred = rf.predict(X_val)
rmse_val = rmse(y_val, y_pred)
print(f"Validation RMSE: {rmse_val:.4f}")

# Context: RMSE of always predicting the training mean.
baseline_rmse = rmse(y_val, np.full_like(y_val, y_train.mean()))
print(f"Constant-mean baseline RMSE: {baseline_rmse:.4f}")

# Pick the closest numerical option (the higher one if exactly between two).
options = [0.1837, 1.837, 18.37, 183.7]
closest = min(options, key=lambda option: (abs(option - rmse_val), -option))
print(f"Closest option: {closest}")

Validation RMSE: 1.8394
Constant-mean baseline RMSE: 2.9600
Closest option: 1.837


<div style="padding: 14px 18px; border-left: 5px solid #16a34a; border-radius: 8px; background: #f0fdf4;">

**Result · Question 2 (random forest baseline)**

A `RandomForestRegressor` with `n_estimators=10` and `random_state=1`, trained on the 6,000 training rows, reaches a validation RMSE of **`1.8394`** MPG on the 2,000 validation rows. Always predicting the training mean gives `2.9600`, so the forest cuts the error by about 38%.

The closest numerical option is **`1.837`** (distance `0.0024`); the next closest, `0.1837`, is far away, so the choice is unambiguous. The result is not an exact match, so the online rule (closest option) is applied. Replaying the same preparation and training code outside the notebook with the course reference versions (Python 3.11.15, NumPy 2.3.3, pandas 2.3.2, scikit-learn 1.7.2) gives `1.8430`, which is even farther from `1.837`, so the library versions do not explain the gap; its cause was not identified.

**Selected option:** **`1.837`**
</div>

<div style="padding: 18px 22px; border-left: 6px solid #2563eb; border-radius: 8px; background: #eff6ff;">

## Step 5 · Question 3: Number of trees

**Objective.** Train random forests with `n_estimators` in `[10, 50, 100, 150]` (`random_state=1`, all other settings fixed), record the validation RMSE for each, and find the tree count with the lowest RMSE at three decimal places. Options: `10`, `50`, `100`, `150`.

</div>

In [9]:
# Train one forest per tree count and record its validation RMSE.
n_estimators_values = [10, 50, 100, 150]
rmse_by_trees = {}

for n_estimators in n_estimators_values:
    forest = RandomForestRegressor(
        n_estimators=n_estimators, random_state=1, n_jobs=-1
    )
    forest.fit(X_train, y_train)
    rmse_by_trees[n_estimators] = rmse(y_val, forest.predict(X_val))

# Tabulate the full-precision RMSE next to its 3-decimal version.
q3_results = pd.DataFrame({
    "n_estimators": list(rmse_by_trees),
    "val_rmse": list(rmse_by_trees.values()),
})
q3_results["val_rmse_3dp"] = q3_results["val_rmse"].round(3)
display(q3_results)

# Compare at 3 decimal places, as the assignment instructs, and detect ties.
best_rmse_3dp = q3_results["val_rmse_3dp"].min()
winners = q3_results.loc[
    q3_results["val_rmse_3dp"] == best_rmse_3dp, "n_estimators"
].tolist()
print(f"Lowest RMSE at 3 decimals: {best_rmse_3dp:.3f}")
print(f"Tree count(s) reaching it: {winners}")

,n_estimators,val_rmse,val_rmse_3dp
0,10,1.839383,1.839
1,50,1.772863,1.773
2,100,1.767643,1.768
3,150,1.768109,1.768


Lowest RMSE at 3 decimals: 1.768
Tree count(s) reaching it: [100, 150]


<div style="padding: 14px 18px; border-left: 5px solid #16a34a; border-radius: 8px; background: #f0fdf4;">

**Result · Question 3 (number of trees)**

| `n_estimators` | Validation RMSE (6 decimals) | Rounded to 3 decimals |
| :---: | ---: | ---: |
| 10 | 1.839383 | 1.839 |
| 50 | 1.772863 | 1.773 |
| 100 | 1.767643 | **1.768** |
| 150 | 1.768109 | **1.768** |

Going from 10 to 100 trees lowers the validation RMSE by about `0.07` MPG, and then the curve flattens: 150 trees is marginally worse than 100 at full precision (`1.768109` vs `1.767643`). The assignment asks to compare at three decimal places, and at that precision **100 and 150 tie at `1.768`** in this environment.

Neither the local nor the online assignment gives a tie-break for tied model scores. The online submission note ("select the higher value" when a result is exactly between two options) refers to a numerical answer equidistant from two options, so it is not a rule for this tie.

**Archived reference-environment replay.** The captured output and reproduction script are preserved in [reference-environment evidence](evidence/homework_06/reference_replay.md). The earlier run of the same preparation and training code outside the notebook with the course reference versions (Python 3.11.15, NumPy 2.3.3, pandas 2.3.2, scikit-learn 1.7.2) gives validation RMSE `1.843032` (10), `1.773650` (50), `1.768922` (100), and `1.768196` (150). At three decimals, 150 (`1.768`) is the unique minimum, ahead of 100 (`1.769`).

**Decision.** `150` is selected because it is the unique minimum at three decimals in the archived reference-environment run. The current environment's tie remains explicitly documented. No tie-break based on choosing the larger forest is applied. The archived replay was not re-executed during the final notebook verification.

**Selected option:** **`150`**
</div>

<div style="padding: 18px 22px; border-left: 6px solid #2563eb; border-radius: 8px; background: #eff6ff;">

## Step 6 · Question 4: Tree depth

**Objective.** For each `max_depth` in `[10, 15, 20, 25]`, train forests with `n_estimators` in `[10, 50, 100, 150]` (`random_state=1`), average the validation RMSE across the four forest sizes, and pick the depth with the lowest mean. Options: `10`, `15`, `20`, `25`.

</div>

In [10]:
# Define the search grid: tree depths and forest sizes.
max_depth_values = [10, 15, 20, 25]
n_estimators_values = [10, 50, 100, 150]

# Train a forest for every (max_depth, n_estimators) pair and record its validation RMSE.
q4_records = []

for max_depth in max_depth_values:
    for n_estimators in n_estimators_values:
        forest = RandomForestRegressor(
            n_estimators=n_estimators,
            max_depth=max_depth,
            random_state=1,
            n_jobs=-1,
        )
        forest.fit(X_train, y_train)
        q4_records.append({
            "max_depth": max_depth,
            "n_estimators": n_estimators,
            "val_rmse": rmse(y_val, forest.predict(X_val)),
        })

q4_results = pd.DataFrame(q4_records)

# Show the full grid as a depth-by-forest-size table.
display(q4_results.pivot(index="max_depth", columns="n_estimators", values="val_rmse"))

# Average the four forest sizes for each depth at full precision.
mean_rmse_by_depth = q4_results.groupby("max_depth")["val_rmse"].mean()
display(mean_rmse_by_depth.to_frame("mean_val_rmse"))

# Select the depth with the lowest mean RMSE and report its margin over the runner-up.
ranked = mean_rmse_by_depth.sort_values()
best_depth = ranked.index[0]
print(f"Lowest mean RMSE: max_depth={best_depth} ({ranked.iloc[0]:.6f})")
print(f"Runner-up: max_depth={ranked.index[1]} ({ranked.iloc[1]:.6f})")
print(f"Margin: {ranked.iloc[1] - ranked.iloc[0]:.6f}")

n_estimators,10,50,100,150
max_depth,,,,
10,1.782096,1.747090,1.743555,1.743790
15,1.829662,1.771177,1.767230,1.766612
20,1.840029,1.774417,1.769159,1.767780
25,1.840011,1.772683,1.766732,1.767272


,mean_val_rmse
max_depth,
10,1.754133
15,1.783670
20,1.787846
25,1.786675


Lowest mean RMSE: max_depth=10 (1.754133)
Runner-up: max_depth=15 (1.783670)
Margin: 0.029537


<div style="padding: 14px 18px; border-left: 5px solid #16a34a; border-radius: 8px; background: #f0fdf4;">

**Result · Question 4 (tree depth)**

| `max_depth` | Mean validation RMSE over `n_estimators` in `[10, 50, 100, 150]` |
| :---: | ---: |
| **10** | **1.754133** |
| 15 | 1.783670 |
| 20 | 1.787846 |
| 25 | 1.786675 |

Depth **10** has the lowest mean RMSE, `0.0295` below the runner-up (depth 15). It also gives the lowest RMSE within each of the four forest sizes, so the choice does not depend on how the sizes are averaged. Deeper trees have higher validation RMSE in this experiment, which is consistent with overfitting; training RMSE was not measured for this comparison.

Each depth's mean uses its four RMSE values at full precision (16 models in total); the selection compares depths by their mean, not by the single best depth and size pair. Depth 10 is the smallest value in the grid, so shallower trees were not evaluated.

**Selected option:** **`10`**
</div>

<div style="padding: 18px 22px; border-left: 6px solid #2563eb; border-radius: 8px; background: #eff6ff;">

## Step 7 · Question 5: Feature importance

**Objective.** Train a `RandomForestRegressor` with `n_estimators=10`, `max_depth=20`, and `random_state=1`, pair `feature_importances_` with the vectorizer feature names, and identify the most important feature among `vehicle_weight`, `horsepower`, `acceleration`, and `engine_displacement`. Explain impurity-based importance and its limitations.

</div>

In [11]:
# Train the forest specified for Q5.
rf_importance = RandomForestRegressor(
    n_estimators=10, max_depth=20, random_state=1, n_jobs=-1
)
rf_importance.fit(X_train, y_train)

# Pair each feature name with its impurity-based importance, largest first.
importances = pd.Series(
    rf_importance.feature_importances_, index=feature_names
).sort_values(ascending=False)
display(importances.to_frame("importance"))
print(f"Importances sum to {importances.sum():.6f}")

# Add up the one-hot columns that come from the same original column.
importance_by_column = importances.groupby(lambda name: name.split("=")[0]).sum()
display(importance_by_column.sort_values(ascending=False).to_frame("importance"))

# Restrict the comparison to the four candidate features.
candidates = ["vehicle_weight", "horsepower", "acceleration", "engine_displacement"]
candidate_importances = importances[candidates].sort_values(ascending=False)
display(candidate_importances.to_frame("importance"))

best_feature = candidate_importances.index[0]
print(f"Most important candidate: {best_feature} ({candidate_importances.iloc[0]:.6f})")

,importance
model_year,0.295915
vehicle_weight,0.203067
fuel_type=Gasoline,0.160712
horsepower,0.078468
engine_displacement,0.061152
acceleration,0.055342
origin=USA,0.042268
drivetrain=Front-wheel drive,0.020108
num_doors,0.019364
fuel_type=Diesel,0.018045


Importances sum to 1.000000


,importance
model_year,0.295915
vehicle_weight,0.203067
fuel_type,0.193589
horsepower,0.078468
engine_displacement,0.061152
acceleration,0.055342
origin,0.052223
drivetrain,0.033653
num_doors,0.019364
num_cylinders,0.007227


,importance
vehicle_weight,0.203067
horsepower,0.078468
engine_displacement,0.061152
acceleration,0.055342


Most important candidate: vehicle_weight (0.203067)


<div style="padding: 14px 18px; border-left: 5px solid #16a34a; border-radius: 8px; background: #f0fdf4;">

**Result · Question 5 (feature importance)**

| Candidate feature | Impurity-based importance |
| :--- | ---: |
| **`vehicle_weight`** | **0.203067** |
| `horsepower` | 0.078468 |
| `engine_displacement` | 0.061152 |
| `acceleration` | 0.055342 |

Among the four candidates, `vehicle_weight` is the most important, about 2.6 times `horsepower`. Across all 16 vectorizer features, `model_year` ranks first (`0.295915`), which agrees with it being the Q1 root split, and `vehicle_weight` is second.

**Reference-environment replay.** Running the same preparation and training code outside the notebook with the course reference versions (Python 3.11.15, NumPy 2.3.3, pandas 2.3.2, scikit-learn 1.7.2) gives `vehicle_weight` `0.203202`, `horsepower` `0.078251`, `engine_displacement` `0.061106`, and `acceleration` `0.055486`. The ranking of the four candidates is the same and each value differs from the table above by less than `0.0003`, so the answer does not depend on the library versions.

**What the number means.** Impurity-based importance (mean decrease in impurity) credits each split's reduction in squared-error impurity, weighted by the proportion of training samples reaching the node, to the feature used for that split. Within each tree, these contributions are summed by feature and normalized to sum to `1`. The forest averages the normalized feature importances across its trees; the resulting importances sum to `1`.

**Limitations.**
- It is computed from the training rows only, so it shows what the trees used to fit the training data, not what generalizes to new data.
- It favors features with many possible thresholds, such as continuous columns, over binary ones.
- Correlated features share credit in an arbitrary way, so a feature can look less important only because a similar one was picked first.
- A categorical feature's importance is distributed across its one-hot columns. For `fuel_type`, their importances are `0.160712`, `0.018045`, and `0.014831`. Their sum (`0.193589`) summarizes the importance assigned to the original categorical feature, but does not remove the biases of impurity-based importance.
- It ranks features but gives no direction or causal effect, and a forest of only 10 trees makes the ranking noisy. Permutation importance on the validation set would be a useful cross-check; it was not computed here.

**Selected option:** **`vehicle_weight`**
</div>

<div style="padding: 18px 22px; border-left: 6px solid #2563eb; border-radius: 8px; background: #eff6ff;">

## Step 8 · Question 6: XGBoost learning rate

**Objective.** Train two independent XGBoost regressors for 100 boosting rounds with the prescribed parameters, changing only `eta` from `0.3` to `0.1`, and compare their validation RMSE. Options: `0.3`, `0.1`, `Both give equal value`.

</div>

In [12]:
# Import XGBoost and report its version, since results can depend on it.
import xgboost as xgb

print(f"XGBoost version: {xgb.__version__}")

# Wrap the sparse matrices in DMatrix objects, keeping the vectorizer's feature names.
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=list(feature_names))
dval = xgb.DMatrix(X_val, label=y_val, feature_names=list(feature_names))

# Evaluate on both sets after every boosting round.
watchlist = [(dtrain, "train"), (dval, "val")]

# Use the prescribed parameters; only eta changes between the two runs.
xgb_params = {
    "eta": 0.3,
    "max_depth": 6,
    "min_child_weight": 1,
    "objective": "reg:squarederror",
    "nthread": 8,
    "seed": 1,
    "verbosity": 1,
}


def train_with_eta(eta):
    # Train for 100 rounds and return the per-round train and validation RMSE.
    evals_result = {}
    xgb.train(
        {**xgb_params, "eta": eta},
        dtrain,
        num_boost_round=100,
        evals=watchlist,
        evals_result=evals_result,
        verbose_eval=False,
    )
    return pd.DataFrame(
        {
            "train_rmse": evals_result["train"]["rmse"],
            "val_rmse": evals_result["val"]["rmse"],
        },
        index=pd.RangeIndex(1, 101, name="round"),
    )


# Run the two independent trainings.
history = {eta: train_with_eta(eta) for eta in [0.3, 0.1]}

# Show both runs at selected rounds.
checkpoints = [1, 10, 25, 50, 75, 100]
display(pd.concat({f"eta={eta}": h.loc[checkpoints] for eta, h in history.items()}, axis=1))

# Compare the final round with the best (minimum) round of each run.
summary = pd.DataFrame({
    eta: {
        "final_val_rmse": h["val_rmse"].iloc[-1],
        "min_val_rmse": h["val_rmse"].min(),
        "min_round": h["val_rmse"].idxmin(),
        "final_train_rmse": h["train_rmse"].iloc[-1],
    }
    for eta, h in history.items()
}).T
display(summary)

best_final = summary["final_val_rmse"].idxmin()
best_min = summary["min_val_rmse"].idxmin()
print(f"Best eta by final-round validation RMSE: {best_final}")
print(f"Best eta by minimum validation RMSE: {best_min}")

XGBoost version: 3.4.1


eta=0.3              eta=0.1          
      train_rmse  val_rmse train_rmse  val_rmse
round                                          
1       2.406533  2.467818   2.747386  2.786354
10      1.487609  1.732816   1.852401  1.969979
25      1.360713  1.743877   1.517628  1.731155
50      1.187329  1.775106   1.405897  1.713127
75      1.031112  1.806859   1.352608  1.720279
100     0.901056  1.826458   1.311918  1.724810

,final_val_rmse,min_val_rmse,min_round,final_train_rmse
0.3,1.826458,1.732048,11.0,0.901056
0.1,1.724810,1.712171,40.0,1.311918


Best eta by final-round validation RMSE: 0.1
Best eta by minimum validation RMSE: 0.1


<div style="padding: 14px 18px; border-left: 5px solid #16a34a; border-radius: 8px; background: #f0fdf4;">

**Result · Question 6 (XGBoost learning rate)**

| `eta` | Validation RMSE at round 100 | Minimum validation RMSE (round) | Training RMSE at round 100 |
| :---: | ---: | ---: | ---: |
| 0.3 | 1.826458 | 1.732048 (11) | 0.901056 |
| **0.1** | **1.724810** | **1.712171 (40)** | 1.311918 |

The assignment does not say whether to compare the final round or the best round of each run, so both were computed. They agree: `eta=0.1` is better at round 100 (`0.1016` lower) and at its best round (`0.0199` lower), so the ambiguity does not change the answer. The two values are not equal, which rules out the third option.

With `eta=0.3` each tree makes a large correction: the validation RMSE bottoms out early (round 11) and then climbs while the training RMSE keeps falling to `0.901056`, a sign of overfitting. With `eta=0.1` the steps are smaller, the validation minimum is lower and arrives later (round 40), and by round 100 it has also started to rise slightly. No early stopping or other tuning was applied, as the assignment instructs.

Both runs are independent trainings of 100 rounds with the prescribed parameters (`max_depth=6`, `min_child_weight=1`, `seed=1`, `nthread=8`); only `eta` changes. These results were obtained with XGBoost `3.4.1`; reproducibility with the course reference version `3.2.0` remains unverified.

**Selected option:** **`0.1`**
</div>

<div style="padding: 18px 22px; border-left: 6px solid #2563eb; border-radius: 8px; background: #eff6ff;">

## Step 9 · Results dashboard and final takeaways

**Objective.** Review the computed evidence, selected options, environment differences, and methodological takeaways for Questions 1-6.

**Answer-selection rule.** The online assignment says to select the closest numerical option and, if the result is exactly between two options, the higher one. The local copy asks for an exact match. The online rule applies when choosing answers.

</div>

**Q3 environment note.** The current environment ties 100 and 150 trees at three decimals. The selected option uses the documented [archived reference replay](evidence/homework_06/reference_replay.md), where 150 is the unique minimum.

<div style="padding: 22px 26px; border: 1px solid #bfdbfe; border-top: 7px solid #2563eb; border-radius: 12px; background: #f8fafc;">

## Homework 6 · Results dashboard

| Question | Focus | Calculated metric / value | Selected option | Brief justification |
| :---: | :--- | :--- | :---: | :--- |
| **Q1** | Decision tree split | Root split: `model_year <= 1997.5`; leaf means `28.614` MPG (2,716 rows) and `31.168` MPG (3,284 rows) | **`model_year`** | The stump splits on the numerical `model_year` column directly, so no one-hot name needed mapping. |
| **Q2** | Random forest baseline validation RMSE | Validation RMSE: `1.8394` MPG (constant-mean baseline: `2.9600`) | **`1.837`** | Closest numerical option under the online rule; `n_estimators=10`, `random_state=1`. |
| **Q3** | Number of trees | Validation RMSE at 3 decimals: `1.839` (10), `1.773` (50), `1.768` (100), `1.768` (150) | **`150`** | Ties with 100 at 3 decimals in this environment; unique minimum (`1.768` vs `1.769`) in the [archived reference replay](evidence/homework_06/reference_replay.md). No model-score tie-break is assumed. |
| **Q4** | Tree depth | Mean validation RMSE: `1.754133` (depth 10), `1.783670` (15), `1.787846` (20), `1.786675` (25) | **`10`** | Lowest mean over the four forest sizes (`0.0295` below depth 15); depth 10 is also lowest at every forest size. |
| **Q5** | Feature importance | Importance: `vehicle_weight` `0.203067`, `horsepower` `0.078468`, `engine_displacement` `0.061152`, `acceleration` `0.055342` | **`vehicle_weight`** | Highest impurity-based importance among the four candidates (10 trees, depth 20); `model_year` is highest overall. |
| **Q6** | XGBoost learning rate | Validation RMSE at round 100: `1.724810` (eta 0.1) vs `1.826458` (eta 0.3) | **`0.1`** | Lower RMSE at round 100 and at the best round (`1.712171` vs `1.732048`); both criteria agree. |

</div>

<div style="margin-top: 16px; padding: 16px 20px; border-left: 5px solid #f59e0b; border-radius: 8px; background: #fffbeb;">

### Final takeaways

- **The random forest outperformed the constant-mean baseline.** A random forest with 10 trees lowers the validation RMSE from `2.9600` (constant-mean baseline) to `1.8394`, about 38%.
- **More trees help, with diminishing returns.** The RMSE falls from `1.839` (10 trees) to `1.768` (100 and 150 trees) and then flattens. The current environment ties 100 and 150 at three decimals; `150` was selected because it is the unique minimum in the documented archived reference run.
- **Shallower trees generalized better here.** Depth 10 has the lowest mean RMSE (`1.754133` vs `1.783670` to `1.787846`) and is best at every forest size. It is the smallest depth tested, so a lower depth might do even better.
- **Importance is a ranking, not an explanation.** `vehicle_weight` leads the four candidates (`0.203067`) and `model_year` leads overall (`0.295915`). Impurity-based importance is computed on training data, favors continuous features, and shares credit among correlated and one-hot columns.
- **A smaller learning rate delayed overfitting in this experiment.** With `eta=0.3` the validation RMSE bottoms out at round 11 (`1.732048`) and rises to `1.826458`, while `eta=0.1` reaches `1.712171` at round 40 and ends at `1.724810`. Both comparison criteria select `0.1`.
- **Check ties and environments.** Q2 does not match an option exactly, and Q3's rounded ranking differs between the current environment and the archived reference run. Q5 identifies the leading candidate in the current environment. Cross-version stability is not assumed. The test set was never used for any decision.

</div>

### Local verification

All 12 code cells were executed in order from a fresh kernel on 2 November 2026 (America/Montevideo), using the project's `.venv`. The run completed without errors and reproduced the results shown above. The dataset checksum was verified before loading. The test set was transformed but was not used for model evaluation or selection.

Environment: Python `3.14.6`, NumPy `2.4.6`, pandas `3.0.6`, scikit-learn `1.9.1`, and XGBoost `3.4.1`. Q3's reference-environment evidence is an archived earlier run, linked above; it was not re-executed in this verification. XGBoost `3.2.0` was not replayed.

**Completion status:** Local analysis and answer review complete. Official submission and grading remain unconfirmed.

### Delivery documentation

The [submission guide](docs/homework_06_submission.md) provides the final answer table, reproduction instructions, environment notes, repository-delivery steps, and a submission record. Review the [Learning in Public draft](docs/homework_06_learning_in_public.md) before publishing and record its URL with the course submission confirmation.